### Remember to install:
    #pip install langchain_community
    #pip install langchain
    #pip install chromadb

In [3]:
!pip install langchain_community langchain chromadb pypdf tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 21.1 MB/s eta 0:00:00


In [5]:
# import libraries
import os
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI
import json
import requests # type: ignore

# Test chatgpt first:

In [7]:
# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("API_KEY") # Loading the API Key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE") # Loading the API Base Url

model_name = "gpt-4o-mini"

# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

# Create a chat completion
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello, how are you. are you alive?"}
    ]
)

# Print the assistant's reply
print(completion.choices[0].message.content)


Hello! I'm just a virtual assistant, so I don't have feelings or a life in the way humans do. But I'm here to help you with any questions or information you might need! How can I assist you today?


# Data Indexing: Single File

In [8]:
# Load PDF:
DOC_PATH = "alphabet_10K_2022.pdf"
CHROMA_PATH = "alphabet_db_name"

# load your pdf doc
loader = PyPDFLoader(DOC_PATH)
pages = loader.load()

# Data indexing folder with pdfs:

In [9]:
! unzip "gravitationalwavefolder-20250404T114250Z-001"

Archive:  gravitationalwavefolder-20250404T114250Z-001.zip
replace gravitationalwavefolder/2503.20778v1.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [11]:
#uploading multiple pdfs:
from glob import glob
from langchain_community.document_loaders import PyPDFLoader

# path to folder with PDFs
DOC_FOLDER = "gravitationalwavefolder/"
pdf_files = glob(DOC_FOLDER + "*.pdf")  # grabs all PDFs in the folder

all_pages = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    all_pages.extend(pages)


In [12]:
print(pdf_files)

['gravitationalwavefolder/2503.20777v1.pdf', 'gravitationalwavefolder/2503.20778v1.pdf', 'gravitationalwavefolder/2503.18887v1.pdf', 'gravitationalwavefolder/2503.18937v1.pdf', 'gravitationalwavefolder/2503.19973v1.pdf']


In [13]:
# Chunking the data
from langchain.text_splitter import RecursiveCharacterTextSplitter # type: ignore

# split the doc into smaller chunks i.e. chunk_size=500
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
#chunks = text_splitter.split_documents(pages)
chunks = text_splitter.split_documents(all_pages)


In [22]:
# Calculate the embeddings and save in database
from langchain.embeddings.openai import OpenAIEmbeddings # type: ignore
from langchain.vectorstores import Chroma # type: ignore

# get OpenAI Embedding model
embeddings = OpenAIEmbeddings(openai_api_key=API_KEY, openai_api_base=OPENAI_API_BASE)

# embed the chunks as vectors and load them into the database
db_chroma = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PATH)

# Data Retrieval from Query:

In [23]:
# this is an example of a user question (query)
query = 'can you use gravitational lensing to study gravitational waves?'

# retrieve context - top 5 most relevant (closests) chunks to the query vector
# (by default Langchain is using cosine distance metric)
docs_chroma = db_chroma.similarity_search_with_score(query, k=15)

# generate an answer based on given user query and retrieved context information
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

# Generate answer with LLM:

In [24]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

# you can use a prompt template
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

# load retrieved context and user query in the prompt template
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)
print(prompt)


Human: 
Answer the question based only on the following context:
test of GR since this theory predicts that GWs travel along geodesics and hence are gravitationally
lensed as they traverse a gravitational field [27]. The detection of a lensed GW will confirm
this property. From a multi-messenger perspective, in the absence of a direct counterpart, EM
information such as comprehensive catalogues of gravitational lenses from surveys such as LSST
and Euclid could provide additional support for candidate lensed GW signals that are found by
GW lensing searches by matching them with their lensed host galaxy (see [69,75,77], for example).
An initial proof-of-concept of such catalogue matching was, for instance, performed in [78] for
some of the ultimately discarded GW lensing candidates from the third GW run. Should a lensed
GW detection be accompanied by lensed EM counterparts (golden objects) we can go a step further
and test whether GWs travel on null geodesics, i.e. whether they propagate

In [25]:
# call LLM model to generate the answer based on the given context and query
model = ChatOpenAI(model_name=model_name, openai_api_key=API_KEY, openai_api_base=OPENAI_API_BASE)
response_text = model.predict(prompt)
print(response_text)

Gravitational lensing can be used to study gravitational waves (GWs) as it enables the detection of lensed GWs, which confirms predictions of General Relativity (GR) that GWs travel along geodesics and are gravitationally lensed as they pass through gravitational fields. The identification of lensed GWs can be enhanced through multi-messenger approaches, where electromagnetic (EM) information from catalogs of gravitational lenses, such as those from LSST and Euclid, supports the identification of candidate lensed GW signals by matching them with their lensed host galaxies.

Additionally, if a lensed GW detection is accompanied by lensed EM counterparts, it allows for further tests of whether GWs travel on null geodesics and propagate at the speed of light. Gravitational lensing also plays a vital role in cosmological studies, particularly in measuring cosmic expansion and understanding dark matter (DM) structures. The presence of gravitational lensing signals can improve the detection 